# Introduction

Libraries:

In [2]:
# DATA WRANGLING, STATISTICS AND VISUALIZATION
import numpy as np
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt
from scipy.stats import randint, uniform, loguniform

# MODEL SELECTION AND OPTIMIZATION
from sklearn.model_selection import train_test_split, RandomizedSearchCV, GridSearchCV

# DATA PREPROCESSING
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer

# CLASSIFICATION MODELS
from sklearn.ensemble import RandomForestClassifier
from sklearn.svm import SVC
from sklearn.neighbors import KNeighborsClassifier

# EVALUATION METRICS
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_curve, roc_auc_score

Assess `accuracy`, `precision`, `recall`, `f1-score`:

In [3]:
# Evaluation metrics considered
metrics = {
    'accuracy': 'accuracy',
    'precision': 'precision',
    'recall': 'recall',
    'f1': 'f1',
    'roc_auc': 'roc_auc'
}

# Number of models trained per search
j = 1000

# Data

In [4]:
# Data file
df_dir = "C:/Users/Miguel/Desarrollo/TFM/data/clean/clinical_data_selected_features.parquet"

# Read the data
df = pd.read_parquet(df_dir)

# Show the first few rows
df.head()

,fa_type,cohort,node,gender,age,cardiac_event,prev_ablation,fa_ablat_time,hbp,atrial_dilation,...,met,diabetes_t2,hypolipidemic,hypercholest,p14,energy,olive_oil,ev_olive_oil,sunflower_oil,alcohol
0,persistent,intervention,Pamplona,female,75.059998,no,no,4.386037,yes,severe,...,4.841667,no,no,no,5.0,2532.445068,0.0,25.0,1.428,0.000000
1,paroxysmal,control,Pamplona,male,73.400002,yes,no,0.476386,no,normal,...,39.271667,no,no,yes,11.0,2185.426025,0.0,25.0,0.000,11.860844
2,paroxysmal,intervention,Pamplona,male,66.010002,no,yes,9.174538,yes,severe,...,17.056667,yes,no,no,9.0,1747.379639,0.0,50.0,0.000,0.681318
3,paroxysmal,intervention,Pamplona,male,70.669998,no,no,0.068446,yes,severe,...,17.056667,no,no,yes,10.0,2519.368896,0.0,25.0,0.000,32.739754
4,paroxysmal,intervention,Pamplona,male,56.090000,no,no,1.776865,no,normal,...,46.959999,no,yes,yes,8.0,2316.215088,0.0,25.0,0.000,10.400000


## Preprocessing

Separate features and target variable:

In [5]:
# Features
X = df.drop("cardiac_event", axis=1)

# Target class manually encoded
y = df["cardiac_event"].map({"no":0, "yes":1})

Divide data set into train, validation and test set:

In [6]:
# Divide into train (and validation) and test set
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    shuffle=True,
    stratify=y
    )

Divide columns according to their data type:

In [7]:
# Numeric features names
numeric_features = X.select_dtypes(include=[np.number]).columns

# Categoric features names
categorical_features = X.select_dtypes(include="category").columns

Preprocessing pipeline:

In [8]:
# Numeric data
numeric_transformer = Pipeline(steps=[
    ('scaler', StandardScaler())
])

# Categoric data
categorical_transformer = Pipeline(steps=[
    ('onehot', OneHotEncoder(handle_unknown='ignore'))
])

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features)
    ]
)

# Random forest

Ensemble the model:

In [9]:
# Full pipeline
pipe_rf = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(random_state=42))
])

# Parameter grid
params_distributions_rf = {
    'classifier__n_estimators': randint(50, 200),
    'classifier__max_depth': [None, 10, 20, 30],
    'classifier__min_samples_split': randint(2, 11)
}

# Random search
random_search_rf = RandomizedSearchCV(
    estimator=pipe_rf,
    param_distributions=params_distributions_rf,
    n_iter=j,
    cv=5,
    scoring=metrics,
    refit = 'roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

Train and optimize the model:

In [10]:
random_search_rf.fit(X_train, y_train)

Fitting 5 folds for each of 1000 candidates, totalling 5000 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__max_depth': [None, 10, ...], 'classifier__min_samples_split': <scipy.stats....001A35FCD5090>, 'classifier__n_estimators': <scipy.stats....001A35FCD8350>}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",1000
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","{'accuracy': 'accuracy', 'f1': 'f1', 'precision': 'precision', 'recall': 'recall', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the spli

Collect the information of every model trained during the optimization:

In [11]:
# Pass the results into a data frame
results_rf = pd.DataFrame(random_search_rf.cv_results_)

# Select relevant colums
relevant_cols = [
    'params',
    'mean_test_accuracy',
    'std_test_accuracy',
    'mean_test_precision',
    'std_test_precision',
    'mean_test_recall',
    'std_test_recall',
    'mean_test_f1',
    'std_test_f1',
    'mean_test_roc_auc',
    'std_test_roc_auc',
    'rank_test_roc_auc'
]

results_summary_rf = results_rf[relevant_cols].sort_values(by='rank_test_roc_auc')

# Show the best 5 models
results_summary_rf.head()

,params,mean_test_accuracy,std_test_accuracy,mean_test_precision,std_test_precision,mean_test_recall,std_test_recall,mean_test_f1,std_test_f1,mean_test_roc_auc,std_test_roc_auc,rank_test_roc_auc
625,"{'classifier__max_depth': None, 'classifier__m...",0.617367,0.022241,0.459139,0.089102,0.182346,0.037477,0.260195,0.049651,0.602631,0.044102,1
396,"{'classifier__max_depth': 30, 'classifier__min...",0.619169,0.016363,0.468137,0.072310,0.177700,0.023045,0.256470,0.029859,0.602014,0.045304,2
452,"{'classifier__max_depth': 10, 'classifier__min...",0.633391,0.042975,0.517326,0.183852,0.196864,0.064251,0.283413,0.090716,0.601916,0.058256,3
35,"{'classifier__max_depth': 30, 'classifier__min...",0.631606,0.030223,0.518546,0.126453,0.206852,0.045180,0.292399,0.055884,0.601911,0.046907,4
982,"{'classifier__max_depth': 30, 'classifier__min...",0.628050,0.032098,0.504799,0.134565,0.197096,0.044012,0.281050,0.059258,0.601855,0.044985,5


# Support Vector Machine


Ensemble the model:

In [12]:
# Full pipeline
pipe_svm = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', SVC(random_state=42))
])

# Parameter grid
params_distributions_svm = {
    'classifier__C': loguniform(1e-3, 1e3),
    'classifier__kernel': ['linear', 'rbf', 'poly', 'sigmoid'],
    'classifier__gamma': ['scale', 'auto'] + list(loguniform(1e-4, 1e1).rvs(10)), 
    'classifier__degree': randint(2, 5),
    'classifier__class_weight': [None, 'balanced']
}

# Random search
random_search_svm = RandomizedSearchCV(
    estimator=pipe_svm,
    param_distributions=params_distributions_svm,
    n_iter=j,
    cv=5,
    scoring=metrics,
    refit = 'roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

Train and optimize the model:

In [13]:
random_search_svm.fit(X_train, y_train)

Fitting 5 folds for each of 1000 candidates, totalling 5000 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...m_state=42))])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__C': <scipy.stats....001A35FE69B90>, 'classifier__class_weight': [None, 'balanced'], 'classifier__degree': <scipy.stats....001A35FE6B1D0>, 'classifier__gamma': ['scale', 'auto', ...], ...}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",1000
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","{'accuracy': 'accuracy', 'f1': 'f1', 'precision': 'precision', 'recall': 'recall', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiat

Collect the information of every model trained during the optimization:

In [14]:
# Pass the results into a data frame
results_svm = pd.DataFrame(random_search_svm.cv_results_)

# Select relevant colums
results_summary_svm = results_svm[relevant_cols].sort_values(by='rank_test_roc_auc')

# Show the best 5 models
results_summary_svm.head()

,params,mean_test_accuracy,std_test_accuracy,mean_test_precision,std_test_precision,mean_test_recall,std_test_recall,mean_test_f1,std_test_f1,mean_test_roc_auc,std_test_roc_auc,rank_test_roc_auc
43,"{'classifier__C': 18.77688470273466, 'classifi...",0.619264,0.038141,0.480515,0.090282,0.249942,0.049115,0.326447,0.056652,0.636859,0.029405,1
475,"{'classifier__C': 41.65510192852256, 'classifi...",0.619264,0.038141,0.480515,0.090282,0.249942,0.049115,0.326447,0.056652,0.636856,0.029541,2
108,"{'classifier__C': 11.561390075135916, 'classif...",0.619264,0.038141,0.480515,0.090282,0.249942,0.049115,0.326447,0.056652,0.636788,0.029411,3
151,"{'classifier__C': 12.834738816570018, 'classif...",0.621049,0.038575,0.486397,0.090402,0.254820,0.044414,0.332292,0.052486,0.636723,0.029278,4
984,"{'classifier__C': 1.905950119667252, 'classifi...",0.621034,0.035337,0.485848,0.081821,0.254704,0.044007,0.331694,0.049375,0.636721,0.029768,5


# K-Nearest Neighbors

Ensemble the model:

In [15]:
# Full pipeline
pipe_knn = Pipeline(steps=[
    ('preprocessor', preprocessor),
    ('classifier', KNeighborsClassifier())
])

# Parameter grid
params_distributions_knn = {
    'classifier__n_neighbors': randint(3, 50),
    'classifier__weights': ['uniform', 'distance'],
    'classifier__p': [1, 2],
    'classifier__algorithm': ['auto', 'ball_tree', 'kd_tree', 'brute']
}

# Random search
random_search_knn = RandomizedSearchCV(
    estimator=pipe_knn,
    param_distributions=params_distributions_knn,
    n_iter=j,
    cv=5,
    scoring=metrics,
    refit='roc_auc',
    n_jobs=-1,
    random_state=42,
    verbose=1
)

Train and optimize the model:

In [16]:
random_search_knn.fit(X_train, y_train)

Fitting 5 folds for each of 1000 candidates, totalling 5000 fits


,"estimator estimator: estimator objectAn object of that type is instantiated for each grid point.This is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...lassifier())])
,"param_distributions param_distributions: dict or list of dictsDictionary with parameters names (`str`) as keys and distributionsor lists of parameters to try. Distributions must provide a ``rvs``method for sampling (such as those from scipy.stats.distributions).If a list is given, it is sampled uniformly.If a list of dicts is given, first a dict is sampled uniformly, andthen a parameter is sampled using that dict as above.","{'classifier__algorithm': ['auto', 'ball_tree', ...], 'classifier__n_neighbors': <scipy.stats....001A35FDD4950>, 'classifier__p': [1, 2], 'classifier__weights': ['uniform', 'distance']}"
,"n_iter n_iter: int, default=10Number of parameter settings that are sampled. n_iter tradesoff runtime vs quality of the solution.",1000
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion ` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.If None, the estimator's score method is used.","{'accuracy': 'accuracy', 'f1': 'f1', 'precision': 'precision', 'recall': 'recall', ...}"
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary `for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given the ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``RandomizedSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`this example`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",'roc_auc'
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- An iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=

Collect the information of every model trained during the optimization:

In [17]:
# Pass the results into a data frame
results_knn = pd.DataFrame(random_search_knn.cv_results_)

# Select relevant colums
results_summary_knn = results_knn[relevant_cols].sort_values(by='rank_test_roc_auc')

# Show the best 5 models
results_summary_knn.head()

,params,mean_test_accuracy,std_test_accuracy,mean_test_precision,std_test_precision,mean_test_recall,std_test_recall,mean_test_f1,std_test_f1,mean_test_roc_auc,std_test_roc_auc,rank_test_roc_auc
723,"{'classifier__algorithm': 'kd_tree', 'classifi...",0.633439,0.009117,0.666667,0.278887,0.043089,0.027608,0.077887,0.044569,0.630363,0.028868,1
548,"{'classifier__algorithm': 'auto', 'classifier_...",0.633439,0.009117,0.666667,0.278887,0.043089,0.027608,0.077887,0.044569,0.630363,0.028868,1
204,"{'classifier__algorithm': 'auto', 'classifier_...",0.633439,0.009117,0.666667,0.278887,0.043089,0.027608,0.077887,0.044569,0.630363,0.028868,1
519,"{'classifier__algorithm': 'auto', 'classifier_...",0.633439,0.009117,0.666667,0.278887,0.043089,0.027608,0.077887,0.044569,0.630363,0.028868,1
830,"{'classifier__algorithm': 'ball_tree', 'classi...",0.633439,0.009117,0.666667,0.278887,0.043089,0.027608,0.077887,0.044569,0.630363,0.028868,1
